# Phase C2 Kaggle Runner
This notebook is entirely standalone. It securely clones the exact branch, runs tests and bounded benchmarks, and orchestrates the Phase C2 empty multi-scale training via the runner script.


In [ ]:
# CONFIGURATION
MODEL_TO_RUN = "M1"  # "M1" (Global-Local) or "M2" (Scalar-Only)
MAX_INTERACTIONS = 150000
RESUME = False
RESUME_BUNDLE_PATH = None  # Provide path if RESUME=True

EXPECTED_BRANCH = "rl-v3-c2-empty-multiscale"
EXPECTED_TAG = "rl-v3-c2-kaggle-v4"

# Exact Hashes
HASH_VALIDATION = "07648ae0c1319124ac52f38f29b960edb63e1c17828e39fccd4e2aac483b4ba4"
HASH_TRAIN_GEN = "6f5e30f6ba6f23432db89a18bb04e070e5fce06d85f58c9c9a17d3d49087d476"
HASH_CONFIG = "da54cc8eb7c98ae666df2060152f1072fbbe101b857f1d065ac6f63f22e28156"
HASH_REWARD = "e9ea675f8d05de842683c2a4d157fa280e9451e651071638ca1848bf9a16e225"
HASH_OBSERVATION = "261ecb534ee33f126baf1d966a4f6fe125b393eee49131ade141382788d0c418"


In [ ]:
import os
import sys
import subprocess
import psutil
import hashlib
from pathlib import Path

# Record Telemetry
print("=== HARDWARE TELEMETRY ===")
print(f"CPU: {psutil.cpu_count(logical=True)} logical cores")
subprocess.run(["lscpu"], check=False)
print("\nGPU:")
subprocess.run(["nvidia-smi", "-L"], check=False)

print("\n=== SOFTWARE TELEMETRY ===")
subprocess.run([sys.executable, "--version"], check=True)
print("\n=== SYSTEM RESOURCES ===")
mem = psutil.virtual_memory()
print(f"RAM: {mem.total / (1024**3):.2f} GB")

# Clone Repo Securely
print("\n=== CLONING REPOSITORY ===")
from kaggle_secrets import UserSecretsClient
try:
    print("Attempting public clone without token...")
    subprocess.run(["git", "clone", "--branch", EXPECTED_BRANCH, "https://github.com/muzzammilsajid1/uav-dynamic-routing.git", "repo"], check=True)
except subprocess.CalledProcessError:
    print("Public clone failed. Attempting with GITHUB_TOKEN secret...")
    user_secrets = UserSecretsClient()
    git_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{git_token}@github.com/muzzammilsajid1/uav-dynamic-routing.git"
    subprocess.run(["git", "clone", "--branch", EXPECTED_BRANCH, repo_url, "repo"], check=True)

os.chdir("repo")

# Remove token from git remote immediately
clean_url = "https://github.com/muzzammilsajid1/uav-dynamic-routing.git"
subprocess.run(["git", "remote", "set-url", "origin", clean_url], check=True)

# Fetch and verify tag
print("\n=== VERIFYING TAG ===")
subprocess.run(["git", "fetch", "origin", "tag", EXPECTED_TAG, "--no-tags"], check=True)
subprocess.run(["git", "checkout", EXPECTED_TAG], check=True)

head_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
tag_commit = subprocess.check_output(["git", "rev-list", "-n", "1", EXPECTED_TAG]).decode().strip()

if head_commit != tag_commit:
    raise ValueError(f"Tag mismatch: HEAD {head_commit} != TAG {tag_commit}")

print(f"Verified Tag: {EXPECTED_TAG} at commit {tag_commit}")

print("\n=== INSTALLING DEPENDENCIES ===")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "cloud/kaggle/requirements_kaggle.txt"], check=True)



In [ ]:
import os
from pathlib import Path

Path("/kaggle/working/uav_phase_c2").mkdir(parents=True, exist_ok=True)

print("\n=== RUNNING EXACT HASH VERIFICATION ===")
def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

files_to_check = {
    "evaluation/manifests/rl_v3_phase_c2_validation.json": HASH_VALIDATION,
    "evaluation/manifests/rl_v3_phase_c2_train_generator.json": HASH_TRAIN_GEN,
    "configs/rl_v3_phase_c2.json": HASH_CONFIG,
    "tools/verification/r2_pb_wrapper.py": HASH_REWARD,
    "rl_v3/observations.py": HASH_OBSERVATION
}

for f, exp in files_to_check.items():
    h = hash_file(f)
    if h != exp:
        raise ValueError(f"Hash mismatch for {f}: Expected {exp}, Got {h}")
print("All hashes matched strictly.")

print("\n=== RUNNING PYTEST ===")
with open("/kaggle/working/uav_phase_c2/pytest_kaggle.log", "w") as pytest_log:
    res = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--basetemp=/kaggle/working/uav_phase_c2/pytest-temp"],
        stdout=pytest_log,
        stderr=subprocess.STDOUT
    )
    if res.returncode != 0:
        with open("/kaggle/working/uav_phase_c2/pytest_kaggle.log", "r") as f:
            print(f.read())
        raise RuntimeError("Pytest failed. Halting pipeline.")
print("Pytest passed successfully.")

print("\n=== RUNNING PREFLIGHT ===")
subprocess.run([sys.executable, "-m", "rl_v3.run_phase_c2", "preflight", "/kaggle/working/uav_phase_c2"], check=True)



In [ ]:
print("\n=== BOUNDED BENCHMARK (2,048 steps) ===")
import time
import torch
import warnings
import gc
warnings.filterwarnings('ignore')

from sb3_contrib import MaskablePPO
from rl_v3.run_phase_c2 import PhaseC2Runner
from rl_v3.phase_c2_env import PhaseC2Env, PhaseC2EndpointGenerator
from tools.verification.r2_pb_wrapper import PotentialShapingWrapper
from rl_v3.run_phase_c2 import M2ScalarWrapper
import json

with open("configs/rl_v3_phase_c2.json") as f:
    config = json.load(f)
    
devices = ["cpu"]
if torch.cuda.is_available():
    devices.append("cuda")

times = {}
for d in devices:
    print(f"Benchmarking {MODEL_TO_RUN} on {d}...")
    torch.manual_seed(42)
    
    gen = PhaseC2EndpointGenerator(seed=42)
    gen.set_active_sizes([15])
    
    env = PhaseC2Env(config, mode="train", generator=gen)
    if config["reward"]["type"] == "R2-PB-empty-v1":
        env = PotentialShapingWrapper(env, gamma=config["reward"]["gamma"], lambda_=config["reward"]["lambda_"])
    
    if MODEL_TO_RUN == "M2":
        env = M2ScalarWrapper(env)
        model = MaskablePPO("MultiInputPolicy", env, device=d, n_steps=2048, batch_size=64)
    else:
        from rl_v3.phase_b_policy import PhaseBFeatureExtractor
        model = MaskablePPO("MultiInputPolicy", env, device=d, n_steps=2048, batch_size=64, policy_kwargs={"features_extractor_class": PhaseBFeatureExtractor, "features_extractor_kwargs": {"features_dim": 256}})
        
    start = time.time()
    model.learn(total_timesteps=2048)
    elapsed = time.time() - start
    times[d] = elapsed
    print(f"{d} took {elapsed:.2f}s (IPS: {2048/elapsed:.1f})")
    
    env.close()
    del model
    del env
    del gen
    if d == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

# Require at least 5% faster to use CUDA
if "cuda" in times and times["cuda"] < times["cpu"] * 0.95:
    best_device = "cuda"
    reason = "CUDA is at least 5% faster than CPU."
else:
    best_device = "cpu"
    reason = "CUDA not significantly faster or unavailable."
    
print(f"\nSelected optimal device: {best_device} ({reason})")

res = {
    "cpu": times.get("cpu"),
    "cuda": times.get("cuda"),
    "selected": best_device,
    "reason": reason
}
with open("/kaggle/working/uav_phase_c2/device_benchmark.json", "w") as f:
    json.dump(res, f, indent=2)



In [ ]:
print(f"\n=== LAUNCHING PHASE C2: {MODEL_TO_RUN} ===")
cmd = [sys.executable, "cloud/kaggle/phase_c2_kaggle_runner.py", "--model", MODEL_TO_RUN, "--interactions", str(MAX_INTERACTIONS), "--device", best_device]
if RESUME:
    if not RESUME_BUNDLE_PATH:
        raise ValueError("RESUME_BUNDLE_PATH must be provided when RESUME=True")
    cmd.extend(["--resume", "--bundle-path", RESUME_BUNDLE_PATH])
    
print("Running command:", " ".join(cmd))
subprocess.run(cmd, check=True)

with open("/kaggle/working/uav_phase_c2/provenance.json", "r") as f:
    prov = json.load(f)
if prov["device"] != best_device:
    raise RuntimeError(f"Device mismatch! Expected {best_device}, Runner used {prov['device']}")

print("\n=== FINAL INVENTORY ===")
subprocess.run(["cat", "/kaggle/working/uav_phase_c2/final_inventory.txt"], check=True)

